# Tarea #1 - Ciencia de Redes (ICD5167- OII467 2026-1)
**Profesor:** Wenceslao Palma <wenceslao.palma@pucv.cl>

## Problema 1: Experimentos con el modelo Erdős-Rényi $G_{n,p}$.
En el modelo Erdős-Rényi, cada par de nodos se une a través de una arista con una probabilidad $p$. Considere un conjunto de nodos $V$ de tamaño $n = 100$. Considere 100 valores para $p$ entre 0 y 1.

(a) Implemente el modelo $G_{n,p}$.
(b) Dibuje grafos para diferentes valores del parámetro $p$ (3-5 ejemplos).
(c) Calcule las métricas requeridas.


In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion estetica
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)


### 1(a) Implementación del modelo $G_{n,p}$ y recolección de métricas


In [ ]:
n = 100
p_values = np.linspace(0, 1, 100)
num_iterations = 30  # Tolerancia estocastica

avg_num_cc = []
avg_largest_cc_size = []
avg_shortest_path = []
avg_density = []
avg_clustering = []

for p in p_values:
    num_cc_iter = []
    largest_cc_size_iter = []
    shortest_path_iter = []
    density_iter = []
    clustering_iter = []
    
    for _ in range(num_iterations):
        G = nx.erdos_renyi_graph(n, p)
        
        num_cc = nx.number_connected_components(G)
        num_cc_iter.append(num_cc)
        
        if num_cc > 0:
            largest_cc_nodes = max(nx.connected_components(G), key=len)
            largest_cc_size_iter.append(len(largest_cc_nodes))
            
            largest_cc = G.subgraph(largest_cc_nodes)
            if len(largest_cc) > 1:
                sp = nx.average_shortest_path_length(largest_cc)
            else:
                sp = 0
            shortest_path_iter.append(sp)
        else:
            largest_cc_size_iter.append(0)
            shortest_path_iter.append(0)
            
        density_iter.append(nx.density(G))
        clustering_iter.append(nx.average_clustering(G))
        
    avg_num_cc.append(np.mean(num_cc_iter))
    avg_largest_cc_size.append(np.mean(largest_cc_size_iter))
    avg_shortest_path.append(np.mean(shortest_path_iter))
    avg_density.append(np.mean(density_iter))
    avg_clustering.append(np.mean(clustering_iter))


### 1(b) Visualización de grafos en diferentes regímenes

Al fijar diferentes probabilidades podemos atravesar los regímenes críticos teóricos donde $\langle k \rangle = p(n-1)$:

1. **Régimen Subcrítico ($p = 0.005$)**: $\langle k \rangle < 1$.
2. **Régimen Crítico ($p = 0.01$)**: $\langle k \rangle \approx 1$. 
3. **Régimen Supercrítico ($p = 0.03$)**: $\langle k \rangle > 1$ pero $\langle k \rangle < \ln(n)$.
4. **Régimen Conectado ($p = 0.1$)**: $\langle k \rangle > \ln(n)$. 


In [ ]:
p_viz = [0.005, 0.01, 0.03, 0.1]
titles = [
    f"Régimen Subcrítico (p={p_viz[0]})\n$<k> \\approx {p_viz[0]*99:.2f}$",
    f"Régimen Crítico (p={p_viz[1]})\n$<k> \\approx {p_viz[1]*99:.2f}$",
    f"Régimen Supercrítico (p={p_viz[2]})\n$<k> \\approx {p_viz[2]*99:.2f}$",
    f"Régimen Conectado (p={p_viz[3]})\n$<k> \\approx {p_viz[3]*99:.2f}$"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for ax, p, title in zip(axes, p_viz, titles):
    G_viz = nx.erdos_renyi_graph(n, p)
    if len(G_viz.nodes) > 0:
        largest_cc = max(nx.connected_components(G_viz), key=len)
        node_colors = ['#FF6F61' if node in largest_cc else '#6B5B95' for node in G_viz.nodes()]
    else:
        node_colors = '#6B5B95'
        
    pos = nx.spring_layout(G_viz, seed=42)
    nx.draw(G_viz, pos, ax=ax, node_color=node_colors, node_size=50, edge_color="gray", alpha=0.6)
    ax.set_title(title, fontsize=14)

plt.tight_layout()
plt.show()


### 1(c) Cálculo y Gráficos Exploratorios

Gráficos sobre el rango $p \in [0, 1]$, y un respectivo zoom iterativo en $p \in [0, 0.1]$ para notar el comportamiento brusco en las franjas analíticas.


In [ ]:
def plot_metric(p_values, metric_values, title, ylabel, show_zoom=True):
    fig, axes = plt.subplots(1, 2 if show_zoom else 1, figsize=(14 if show_zoom else 7, 5))
    
    if not show_zoom:
        axes = [axes]
        
    sns.lineplot(x=p_values, y=metric_values, ax=axes[0], color="#005b96", linewidth=2.5)
    axes[0].set_title(f"{title} (Rango Completo)", fontsize=14)
    axes[0].set_xlabel("Probabilidad de enlace $p$")
    axes[0].set_ylabel(ylabel)
    
    if show_zoom:
        zoom_mask = p_values <= 0.1
        sns.lineplot(x=p_values[zoom_mask], y=np.array(metric_values)[zoom_mask], ax=axes[1], color="#d64161", linewidth=2.5)
        axes[1].axvline(x=0.01, color='green', linestyle='--', label='$p=0.01$ (Crítico)')
        axes[1].axvline(x=np.log(n)/n, color='orange', linestyle='--', label='$p_{conectado} \\approx \\ln(100)/100$')
        axes[1].set_title(f"Zoom In: Transición de Fase ($p \\in [0, 0.1]$)", fontsize=14)
        axes[1].set_xlabel("Probabilidad de enlace $p$")
        axes[1].set_ylabel(ylabel)
        axes[1].legend()

    plt.tight_layout()
    plt.show()

# Despliegue de gráficos
plot_metric(p_values, avg_num_cc, "Número de Componentes Conexas", "Cantidad de CC", show_zoom=True)
plot_metric(p_values, avg_largest_cc_size, "Tamaño de la Componente Gigante", "Número de Nodos", show_zoom=True)
plot_metric(p_values, avg_shortest_path, "Camino Mínimo Promedio (Comp. Gigante)", "Longitud del Camino", show_zoom=True)
plot_metric(p_values, avg_density, "Densidad de la Red", "Densidad", show_zoom=True)
plot_metric(p_values, avg_clustering, "Coeficiente de Clustering Promedio", "Coeficiente $C$", show_zoom=True)


### 1(c.6) Conclusiones
El comportamiento analizado con las métricas respeta rigurosamente lo estudiado teóricamente:

1. **Aparición de la Componente Gigante (Fase Crítica):** El número de componentes de la red se resetea a 1 rápidamente para asimilar el surgimiento de la componente mayor que va succionando toda la materia. Esto puede observarse en los gráficos con *zoom*, el punto de quiebre ocurre con total claridad pasando $p \approx 0.01$ ($\langle k \rangle = 1$). A partir del umbral $p_{conectado} \approx \frac{\ln(100)}{100} \approx 0.046$, el volumen de la componente mayor se equipara formalmente a $N$ y desaparecen los nodos aislados en la red aleatoria.
2. **Forma del Camino Promedio:** Inmediatamente después del punto crítico, observamos un "pic" global en el camino mínimo promedio. En el preciso momento en que todos los componentes entran por primera vez a este clúster mayor, tienen distancias exageradas produciendo una estructura larga como una red en fila/filamento; una vez se aumenta $p$, el mundo pequeño reordena las distancias bajando drásticamente su tamaño.
3. **Propiedades Lineales de Densidad y Clustering:** La evolución de densidad y clustering crecen de manera lineal e idéntica a la propia probabilidad $p$.


## Problema 2: Erdős-Rényi $G_{n,p}$ y los valores de $k_{min}$ y $k_{max}$


Pendiente de implementación con datos locales.


## Problema 3: Experimentos con una red real


Pendiente de implementación con datos locales.
